In [2]:
# CELL 1 — INSTALL
!pip install transformers torch Pillow tqdm -q

In [3]:
# CELL 2 — IMPORTS
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import shutil

In [4]:
# CELL 3 — PATHS
CAPTIONS_CSV  = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/captions_new.csv"
GALLERY_DIR   = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/split_images/gallery"
QUERY_DIR     = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/split_images/query"

OUTPUT_DIR    = "/kaggle/working/embeddings"
VIS_GAL_DIR   = os.path.join(OUTPUT_DIR, "visual", "gallery")
VIS_QRY_DIR   = os.path.join(OUTPUT_DIR, "visual", "query")
CAP_GAL_DIR   = os.path.join(OUTPUT_DIR, "caption", "gallery")

for d in [VIS_GAL_DIR, VIS_QRY_DIR, CAP_GAL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Captions CSV exists :", os.path.exists(CAPTIONS_CSV))
print("Gallery dir exists  :", os.path.exists(GALLERY_DIR))
print("Query dir exists    :", os.path.exists(QUERY_DIR))
print("Gallery images      :", len(os.listdir(GALLERY_DIR)))
print("Query images        :", len(os.listdir(QUERY_DIR)))

Captions CSV exists : True
Gallery dir exists  : True
Query dir exists    : True
Gallery images      : 12612
Query images        : 14218


In [5]:
# CELL 4 — LOAD CLIP
MODEL_ID  = "openai/clip-vit-base-patch32"
device    = "cuda" if torch.cuda.is_available() else "cpu"

processor = CLIPProcessor.from_pretrained(MODEL_ID)
model     = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print(f"CLIP loaded on {device}")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded on cuda


In [6]:
# CELL 5 — LOAD CAPTIONS CSV
captions_df = pd.read_csv(CAPTIONS_CSV)
print(f"Total captions: {len(captions_df)}")
print(captions_df["split"].value_counts())

# Build lookup: filename -> caption
captions_df["filename"] = captions_df["cropped_image_path"].apply(os.path.basename)
caption_lookup = dict(zip(captions_df["filename"], captions_df["caption"]))
item_id_lookup = dict(zip(captions_df["filename"], captions_df["item_id"]))
print(f"Caption lookup built: {len(caption_lookup)} entries")

Total captions: 52712
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64
Caption lookup built: 52712 entries


In [14]:
# CELL 6 — VISUAL EMBEDDING FUNCTION
BATCH_SIZE = 64

def embed_images(image_paths):
    images = []
    for p in image_paths:
        try:
            images.append(Image.open(p).convert("RGB"))
        except Exception:
            images.append(Image.new("RGB", (224, 224)))

    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = model.vision_model(pixel_values=inputs["pixel_values"].to(device))
        feats = feats.pooler_output
        feats = model.visual_projection(feats)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()


def embed_texts(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        feats = model.text_model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device)
        )
        feats = feats.pooler_output
        feats = model.text_projection(feats)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

In [15]:
# CELL 7 — GALLERY VISUAL EMBEDDINGS
gallery_files = sorted(os.listdir(GALLERY_DIR))
print(f"Gallery images: {len(gallery_files)}")

all_embeddings = []
all_filenames  = []
all_item_ids   = []

for start in tqdm(range(0, len(gallery_files), BATCH_SIZE), desc="Gallery visual"):
    batch_files = gallery_files[start : start + BATCH_SIZE]
    batch_paths = [os.path.join(GALLERY_DIR, f) for f in batch_files]
    embs        = embed_images(batch_paths)

    all_embeddings.append(embs)
    all_filenames.extend(batch_files)
    all_item_ids.extend([item_id_lookup.get(f, "") for f in batch_files])

gallery_visual_embs = np.vstack(all_embeddings)
gallery_filenames   = np.array(all_filenames)
gallery_item_ids    = np.array(all_item_ids)

np.save(os.path.join(VIS_GAL_DIR, "embeddings.npy"),  gallery_visual_embs)
np.save(os.path.join(VIS_GAL_DIR, "filenames.npy"),   gallery_filenames)
np.save(os.path.join(VIS_GAL_DIR, "item_ids.npy"),    gallery_item_ids)

print(f"Gallery visual embeddings: {gallery_visual_embs.shape}")
print(f"Saved to {VIS_GAL_DIR}")

Gallery images: 12612


Gallery visual: 100%|██████████| 198/198 [02:07<00:00,  1.55it/s]

Gallery visual embeddings: (12612, 512)
Saved to /kaggle/working/embeddings/visual/gallery


In [16]:
# CELL 8 — GALLERY CAPTION EMBEDDINGS
all_cap_embeddings = []
all_cap_filenames  = []
missing_captions   = 0

for start in tqdm(range(0, len(gallery_files), BATCH_SIZE), desc="Gallery caption"):
    batch_files    = gallery_files[start : start + BATCH_SIZE]
    batch_captions = []
    for f in batch_files:
        cap = caption_lookup.get(f, "a clothing item")
        if not isinstance(cap, str) or cap.strip() == "":
            cap = "a clothing item"
            missing_captions += 1
        batch_captions.append(cap)

    embs = embed_texts(batch_captions)
    all_cap_embeddings.append(embs)
    all_cap_filenames.extend(batch_files)

gallery_caption_embs = np.vstack(all_cap_embeddings)
gallery_cap_filenames = np.array(all_cap_filenames)

np.save(os.path.join(CAP_GAL_DIR, "embeddings.npy"), gallery_caption_embs)
np.save(os.path.join(CAP_GAL_DIR, "filenames.npy"),  gallery_cap_filenames)

print(f"Gallery caption embeddings: {gallery_caption_embs.shape}")
print(f"Missing captions fallback : {missing_captions}")
print(f"Saved to {CAP_GAL_DIR}")

Gallery caption: 100%|██████████| 198/198 [00:05<00:00, 35.21it/s]

Gallery caption embeddings: (12612, 512)
Missing captions fallback : 1
Saved to /kaggle/working/embeddings/caption/gallery


In [17]:
# CELL 9 — QUERY VISUAL EMBEDDINGS
query_files = sorted(os.listdir(QUERY_DIR))
print(f"Query images: {len(query_files)}")

all_q_embeddings = []
all_q_filenames  = []
all_q_item_ids   = []

for start in tqdm(range(0, len(query_files), BATCH_SIZE), desc="Query visual"):
    batch_files = query_files[start : start + BATCH_SIZE]
    batch_paths = [os.path.join(QUERY_DIR, f) for f in batch_files]
    embs        = embed_images(batch_paths)

    all_q_embeddings.append(embs)
    all_q_filenames.extend(batch_files)
    all_q_item_ids.extend([item_id_lookup.get(f, "") for f in batch_files])

query_visual_embs = np.vstack(all_q_embeddings)
query_filenames   = np.array(all_q_filenames)
query_item_ids    = np.array(all_q_item_ids)

np.save(os.path.join(VIS_QRY_DIR, "embeddings.npy"), query_visual_embs)
np.save(os.path.join(VIS_QRY_DIR, "filenames.npy"),  query_filenames)
np.save(os.path.join(VIS_QRY_DIR, "item_ids.npy"),   query_item_ids)

print(f"Query visual embeddings: {query_visual_embs.shape}")
print(f"Saved to {VIS_QRY_DIR}")

Query images: 14218


Query visual: 100%|██████████| 223/223 [02:23<00:00,  1.55it/s]

Query visual embeddings: (14218, 512)
Saved to /kaggle/working/embeddings/visual/query


In [18]:
# CELL 10 — SANITY CHECK
print("=== SANITY CHECK ===")
print(f"Gallery visual : {gallery_visual_embs.shape}   expected (12612, 512)")
print(f"Gallery caption: {gallery_caption_embs.shape}  expected (12612, 512)")
print(f"Query visual   : {query_visual_embs.shape}     expected (14218, 512)")
print()
# Verify filenames match between visual and caption gallery
match = (gallery_filenames == gallery_cap_filenames).all()
print(f"Gallery visual/caption filename order match: {match}")
print()
# Check norms are 1
norms = np.linalg.norm(gallery_visual_embs[:5], axis=1)
print(f"Sample L2 norms (should be ~1.0): {norms}")

=== SANITY CHECK ===
Gallery visual : (12612, 512)   expected (12612, 512)
Gallery caption: (12612, 512)  expected (12612, 512)
Query visual   : (14218, 512)     expected (14218, 512)

Gallery visual/caption filename order match: True

Sample L2 norms (should be ~1.0): [1.         0.99999994 1.         1.         1.        ]


In [19]:
# CELL 11 — ZIP AND SAVE
shutil.make_archive("/kaggle/working/embeddings", "zip", "/kaggle/working/embeddings")
size_mb = os.path.getsize("/kaggle/working/embeddings.zip") / (1024**2)
print(f"ZIP: /kaggle/working/embeddings.zip  ({size_mb:.1f} MB)")

ZIP: /kaggle/working/embeddings.zip  (67.0 MB)
